# 结构化掩码可视化：Cls / ImageNet（SwinV2-T + EDEN）
**回答的问题：各训练阶段的 ckpt 当前会怎么蒙。**

口径与 seg/det/reid 三臂完全一致（measure_mask_viz 是任务无关的，只有
建模/取数/task loss 由 cls_probe 与本 notebook 的 grad_fn 适配）：
G = |∂L_task/∂x₀|（task loss = swin head 的 CE）-> 划块均值 -> softmax 概率
-> Gumbel-top-k 现采 -> 蒙后输入 x̃₀。CRN：跨 ckpt 同图同噪声，严格可比。

需在 Swin-Transformer 训练环境下运行（建模/取数走 cls_probe，
Swin 目录会被插到 sys.path 最前——其 config/models/data 模块名与常见包重名）。

In [ ]:
import os, sys
import torch
import torch.nn.functional as F

VISUAL = os.path.abspath('.')          # 内核启动时 cwd 就是 visual/
if VISUAL not in sys.path:
    sys.path.insert(0, VISUAL)

import importlib
import probe_core as pc
import cls_probe as cp
importlib.reload(pc)
importlib.reload(cp)
assert getattr(pc, 'MASK_VIZ_VERSION', 0) >= 3, \
    'probe_core 是旧版（无 MASK_VIZ_VERSION>=2 的掩码可视化）——请确认 visual/probe_core.py 已同步并重启内核'

In [ ]:
# ---- 配置 ----
import glob, re
SWIN = os.path.abspath('../tasks/Classification/imagenet/Swin-Transformer')
CFG = os.path.join(SWIN, 'configs/swinv2/swinv2_tiny_patch4_window8_256_eden_tweight_mask.yaml')
IMAGENET_ROOT = '<改成 imagenet 根目录>'   # 与训练 --data-path 一致
CKPTS = sorted(glob.glob(os.path.join(SWIN, 'output/**/ckpt_epoch_*.pth'), recursive=True),
               key=lambda p: int(re.findall(r'\d+', os.path.basename(p))[-1]))
OUT = os.path.join(VISUAL, 'outputs/mask_viz/cls')
NUM_IMAGES = 8
DISPLAY_IMAGES = 4
MASK_RATIO = None     # None = 模型配置值
GUIDED = True         # True = G 引导选址；False = 预热期均匀随机
SEED = 0
os.makedirs(OUT, exist_ok=True)
CKPTS

In [ ]:
# 数据只取一次，所有 ckpt 共用（同图同分布，跨 ckpt 可比）
# swin val 归一化：ToTensor([0,1]) 后 Normalize(0.485/0.229)，反归一化 scale=1.0
model, config = cp.build_cls_model(CFG, None, data_path=IMAGENET_ROOT)
batch = cp.build_cls_val_batch(model, config, NUM_IMAGES)
batch['img01'] = pc.img01_from_xis(batch['xis'],
                                   mean=[0.485, 0.456, 0.406],
                                   std=[0.229, 0.224, 0.225], scale=1.0)
del model; torch.cuda.empty_cache()

In [ ]:
results, labels = [], []
try:
    import matplotlib  # noqa: F401
    _HAS_MPL = True
except ImportError:
    _HAS_MPL = False
    print('[warn] matplotlib not available: skipping plots, npz caches still saved '
          '(mask_montage_all.py is unaffected)')
for ckpt in CKPTS:
    model, _ = cp.build_cls_model(CFG, ckpt, data_path=IMAGENET_ROOT)
    tag = os.path.splitext(os.path.basename(ckpt))[0]
    # cls 整批一次反传（val 同尺寸）：task loss = forward_features + head 的 CE
    def grad_fn(X, dsis):
        ys = torch.as_tensor([int(d) for d in dsis], device=X.device)
        logits = model.model.head(model.model.forward_features(X))
        return F.cross_entropy(logits, ys)
    res = pc.measure_mask_viz(model, batch, grad_fn, batch_grad=True,
                              mask_ratio=MASK_RATIO, guided=GUIDED, seed=SEED)
    pc.save_mask_viz_cache(os.path.join(OUT, f'{tag}_cache.npz'), res)
    if _HAS_MPL:
        fig = pc.plot_mask_viz(res, title=tag, max_images=DISPLAY_IMAGES,
                               save_path=os.path.join(OUT, f'{tag}_mask_viz.png'))
    results.append(res); labels.append(tag)
    print(f'{tag}: masked={res["masked_frac"].mean():.2f} '
          f'(k={res["k"]}/{res["g"]**2}, alpha={res["alpha"]:g})')
    del model; torch.cuda.empty_cache()
if _HAS_MPL:
    fig  # 显示最后一张

In [ ]:
# 跨 ckpt 演化拼图：行 = 训练阶段，列 = 同一批图（无 matplotlib 时跳过）
if _HAS_MPL:
    for mode in ['masked', 'prob', 'g']:
        pc.plot_mask_viz_evolution(results, labels, mode=mode,
                                   max_images=DISPLAY_IMAGES,
                                   save_path=os.path.join(OUT, f'evolution_{mode}.png'))
    print('saved evolution_{masked,prob,g}.png')
else:
    print('skipped (no matplotlib); npz caches are complete')

In [ ]:
# 免重测复画：从 npz 缓存重建 res（换 DISPLAY_IMAGES / 标题都行；掩码已在缓存里）
if _HAS_MPL:
    REPLAY_TAG = os.path.splitext(os.path.basename(CKPTS[-1]))[0]  # 改成想复画的 ckpt tag
    res = pc.load_mask_viz_cache(os.path.join(OUT, f'{REPLAY_TAG}_cache.npz'))
    fig = pc.plot_mask_viz(res, title=f'{REPLAY_TAG} (replay)',
                           max_images=DISPLAY_IMAGES,
                           save_path=os.path.join(OUT, f'{REPLAY_TAG}_replay.png'))
    fig
else:
    print('skipped (no matplotlib)')